# AI 부동산 매수 컨설턴트 — Gradio + OpenAI

**현장 데이터 디지털화 과정 · Day05 코랩 실습 예시 (프롬프트로 만든 예제)**

예산 · 소득 · 조건을 넣으면 지역 추천 → 실거래 중앙값 · 매물 비교 → 매수 절차 → 단계별 비용 · 자금 계획까지 5페이지로 보여 줍니다.

### 시작하는 방법
1. 위쪽 **「Drive로 복사」** 로 내 드라이브에 사본을 만듭니다.
2. 왼쪽 열쇠(**보안 비밀**)에 `OPENAI_API_KEY` 를 넣거나, 챗봇 화면 맨 위 칸에 붙여 넣습니다.
3. 아래 코드 셀 **하나**를 실행합니다(설치부터 실행까지 한 번에, 1~2분). `https://….gradio.live` 링크를 누르면 크게 열립니다.
4. 화면의 1 → 5페이지 순서로 진행합니다. 5페이지 숫자를 바꾸면 비용표가 바로 다시 계산되고, 맨 아래에서 엑셀로 받습니다.

### 실거래가를 보려면 — 국토교통부 API 키(무료)
1. 공공데이터포털 https://www.data.go.kr 로그인 → **「국토교통부_아파트 매매 실거래가 상세 자료」** 검색 → 활용신청.
2. 마이페이지 → 개발계정에서 **일반 인증키(Decoding)** 를 복사해 화면의 「국토교통부 실거래가 API 키」 칸에 넣습니다.
3. 승인 직후에는 몇 시간 동안 키가 안 될 수 있습니다. 그때는 실거래 칸이 비고, 화면에 실패 이유가 나옵니다.

### 프롬프트에서 그대로 못 한 것 (지어내지 않기 위해)
| 프롬프트 요청 | 이 예제에서 한 방법 | 이유 |
|---|---|---|
| 네이버부동산 매물 자동 수집 | 내가 직접 확인한 매물을 **붙여 넣기**(단지 · 면적 · 층 · 호가 · 링크 · 확인 날짜). 없으면 「매물 확인 불가」 | 매물 사이트는 자동 수집을 막고 이용약관으로 금지합니다. AI 가 매물을 만들어 내면 안 됩니다 |
| 실거래가 조사 | 국토교통부 공식 API 로 최근 12개월을 받아 **파이썬이** 3 · 6 · 12개월 중앙값 계산(해제 거래 제외) | 공식 자료 · 계산은 AI 가 아니라 코드로 |
| 규제지역 확인 | 사용자가 체크(토지이음에서 확인) + 선택하면 OpenAI 웹 검색으로 출처 링크와 함께 확인. 검색이 안 되면 「데이터 접근 제한」 | 규제는 수시로 바뀌어 코드에 박아 두면 틀립니다 |
| LTV · DSR 최신 규제 | 기본값(2025-10-15 대책 기준)을 화면에서 바꿀 수 있게 하고 【확실하지 않은 사실】 표시 | 대출 규제는 발표마다 바뀝니다 |
| 출퇴근 시간 | 지역 중심 사이 직선거리로 어림 | 실제 경로 API 는 별도 키가 필요합니다 |

**표시 규칙**: AI 가 낸 지역 장단점 · 점수는 【ChatGPT의 의견】, 확인 못 한 내용은 【확실하지 않은 사실】. 이 결과는 참고용이며 투자 권유가 아닙니다.

In [ ]:
!pip install -q -U gradio openai plotly pandas openpyxl

# ============================================================
# AI 부동산 매수 컨설턴트 (Gradio 6 + OpenAI)
#  - 계산(대출 · 세금 · 비용 · 중앙값)은 모두 파이썬이 합니다.
#  - AI(ChatGPT)는 지역의 장단점 · 점수 같은 「의견」만 냅니다.
#  - 실거래가는 국토교통부 공공데이터 API(키 필요)로만 가져옵니다.
#  - 매물(호가)은 사용자가 확인한 매물을 붙여 넣은 것만 씁니다. 지어내지 않습니다.
# ============================================================
import os, io, re, json, math, inspect, datetime as dt, urllib.parse, urllib.request
import xml.etree.ElementTree as ET
from statistics import median
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import gradio as gr
from openai import OpenAI

TODAY = dt.date.today()
MODEL = "gpt-4.1-mini"

# ---------- 지역 목록: 시군구 코드(국토교통부 실거래가 조회용) · 대략적인 중심 좌표 ----------
# 좌표는 지도 표시와 직선거리 계산용 어림값입니다.
REGIONS = {
    "서울 종로구": ("11110", 37.573, 126.979), "서울 중구": ("11140", 37.564, 126.998),
    "서울 용산구": ("11170", 37.532, 126.990), "서울 성동구": ("11200", 37.563, 127.037),
    "서울 광진구": ("11215", 37.538, 127.082), "서울 동대문구": ("11230", 37.574, 127.040),
    "서울 중랑구": ("11260", 37.606, 127.093), "서울 성북구": ("11290", 37.589, 127.017),
    "서울 강북구": ("11305", 37.640, 127.026), "서울 도봉구": ("11320", 37.669, 127.047),
    "서울 노원구": ("11350", 37.654, 127.057), "서울 은평구": ("11380", 37.603, 126.929),
    "서울 서대문구": ("11410", 37.579, 126.937), "서울 마포구": ("11440", 37.566, 126.902),
    "서울 양천구": ("11470", 37.517, 126.867), "서울 강서구": ("11500", 37.551, 126.850),
    "서울 구로구": ("11530", 37.495, 126.888), "서울 금천구": ("11545", 37.457, 126.895),
    "서울 영등포구": ("11560", 37.526, 126.896), "서울 동작구": ("11590", 37.512, 126.940),
    "서울 관악구": ("11620", 37.478, 126.952), "서울 서초구": ("11650", 37.484, 127.033),
    "서울 강남구": ("11680", 37.517, 127.047), "서울 송파구": ("11710", 37.515, 127.106),
    "서울 강동구": ("11740", 37.530, 127.124),
    "경기 성남시 분당구": ("41135", 37.383, 127.119), "경기 성남시 수정구": ("41131", 37.450, 127.146),
    "경기 성남시 중원구": ("41133", 37.430, 127.137), "경기 수원시 영통구": ("41117", 37.260, 127.047),
    "경기 수원시 장안구": ("41111", 37.304, 127.010), "경기 수원시 권선구": ("41113", 37.258, 126.972),
    "경기 수원시 팔달구": ("41115", 37.282, 127.020), "경기 용인시 수지구": ("41465", 37.322, 127.098),
    "경기 용인시 기흥구": ("41463", 37.280, 127.115), "경기 용인시 처인구": ("41461", 37.234, 127.201),
    "경기 화성시": ("41590", 37.200, 127.073), "경기 광명시": ("41210", 37.479, 126.865),
    "경기 하남시": ("41450", 37.539, 127.215), "경기 과천시": ("41290", 37.429, 126.988),
    "경기 안양시 동안구": ("41173", 37.392, 126.951), "경기 안양시 만안구": ("41171", 37.387, 126.932),
    "경기 의왕시": ("41430", 37.345, 126.968), "경기 군포시": ("41410", 37.362, 126.935),
    "경기 고양시 일산동구": ("41285", 37.659, 126.775), "경기 고양시 일산서구": ("41287", 37.675, 126.751),
    "경기 고양시 덕양구": ("41281", 37.638, 126.832), "경기 남양주시": ("41360", 37.636, 127.216),
    "경기 구리시": ("41310", 37.594, 127.130), "경기 김포시": ("41570", 37.615, 126.716),
    "경기 평택시": ("41220", 36.992, 127.113), "경기 의정부시": ("41150", 37.738, 127.034),
    "인천 연수구": ("28185", 37.410, 126.678), "인천 부평구": ("28237", 37.507, 126.722),
    "인천 남동구": ("28200", 37.447, 126.731), "인천 서구": ("28260", 37.545, 126.676),
}
REGION_NAMES = list(REGIONS)
FACTORS = ["자연환경", "학군", "직주근접", "교통", "생활편의", "미래가치"]

OPINION = "【ChatGPT의 의견】"
UNSURE = "【확실하지 않은 사실】"

# ---------- 공통 도우미 ----------
def man(won_man):          # 만원 → 「12억 3,400만」
    v = int(round(won_man))
    eok, rest = divmod(abs(v), 10000)
    s = (f"{eok}억 " if eok else "") + (f"{rest:,}만" if rest else ("" if eok else "0만"))
    return ("-" if v < 0 else "") + s.strip() + "원"

def km(a, b):
    la1, lo1, la2, lo2 = map(math.radians, (a[0], a[1], b[0], b[1]))
    h = math.sin((la2-la1)/2)**2 + math.cos(la1)*math.cos(la2)*math.sin((lo2-lo1)/2)**2
    return 6371 * 2 * math.asin(math.sqrt(h))

def commute_min(region, work):
    # 직선거리 × 1.4(실제 길) ÷ 평균 25km/h + 환승·도보 15분 — 어림값입니다
    d = km(REGIONS[region][1:], REGIONS[work][1:])
    return round(d * 1.4 / 25 * 60 + 15)

def get_key(box):
    k = (box or "").strip() or os.environ.get("OPENAI_API_KEY", "")
    if not k:
        raise gr.Error("OpenAI API 키가 없습니다. 맨 위 「OpenAI API 키」 칸에 붙여 넣거나 코랩 보안 비밀 OPENAI_API_KEY 를 넣어 주세요.")
    return OpenAI(api_key=k)

def num(x, default=0.0):
    try:
        return float(str(x).replace(",", "").strip())
    except Exception:
        return default


# ---------- 1. 대출 가능액 (파이썬 계산) ----------
def monthly_payment(principal, rate_pct, years):
    r, n = rate_pct / 100 / 12, years * 12
    if principal <= 0:
        return 0.0
    return principal / n if r == 0 else principal * r / (1 - (1 + r) ** -n)

def max_loan(price, income, exist_yearly, regulated, first_buyer, ltv_reg, ltv_free, ltv_first,
             dsr, rate, stress, years, cap_on):
    """price · income · 상환액은 만원 단위. 세 가지 한도 중 가장 작은 값."""
    ltv = (ltv_first if first_buyer else (ltv_reg if regulated else ltv_free)) / 100
    by_ltv = price * ltv
    # DSR: (새 대출 연 원리금 + 기존 대출 연 원리금) ≤ 연소득 × DSR. 금리는 스트레스 금리를 더해 잽니다
    room = income * dsr / 100 - exist_yearly
    r, n = (rate + stress) / 100 / 12, years * 12
    by_dsr = 0.0 if room <= 0 else (room / 12) * ((1 - (1 + r) ** -n) / r if r else n)
    by_cap = float("inf")
    if cap_on:                       # 수도권 · 규제지역 주택담보대출 금액 한도
        by_cap = 60000 if price <= 150000 else (40000 if price <= 250000 else 20000)
    loan = max(0.0, min(by_ltv, by_dsr, by_cap))
    detail = pd.DataFrame([
        ["LTV 한도", f"{ltv*100:.0f}%", man(by_ltv)],
        ["DSR 한도", f"{dsr:.0f}% · 금리 {rate}+{stress}%p · {years}년", man(by_dsr)],
        ["금액 한도", "켬" if cap_on else "끔", "없음" if by_cap == float("inf") else man(by_cap)],
        ["예상 대출 가능액", "가장 작은 값", man(loan)],
    ], columns=["기준", "조건", "금액"])
    return loan, detail


# ---------- 2. 세금 · 수수료 (파이썬 계산) ----------
def acquisition_tax(price, houses_after, regulated, area):
    """주택 취득세 + 지방교육세 + 농어촌특별세 (만원). 감면은 넣지 않았습니다."""
    if houses_after >= 4 or (houses_after == 3 and regulated):
        rate = 12.0
    elif houses_after == 3 or (houses_after == 2 and regulated):
        rate = 8.0
    else:                                             # 1주택 또는 비조정 2주택
        if price <= 60000:
            rate = 1.0
        elif price <= 90000:
            rate = round((price / 10000 * 2 / 3 - 3), 2)
        else:
            rate = 3.0
    tax = price * rate / 100
    edu = price * (0.4 if rate >= 8 else rate * 0.1) / 100
    big = area > 85
    farm = price * ((1.0 if rate >= 12 else 0.6 if rate >= 8 else 0.2) if big else 0) / 100
    return tax + edu + farm, rate

def brokerage(price, vat):
    # 매매 중개보수 상한 요율 (2021-10-19 시행 기준)
    if price < 5000:
        fee = min(price * 0.006, 25)
    elif price < 20000:
        fee = min(price * 0.005, 80)
    elif price < 90000:
        fee = price * 0.004
    elif price < 120000:
        fee = price * 0.005
    elif price < 150000:
        fee = price * 0.006
    else:
        fee = price * 0.007
    return fee * (1.1 if vat else 1.0)


# ---------- 3. 국토교통부 아파트 매매 실거래가 ----------
API_URLS = [
    "https://apis.data.go.kr/1613000/RTMSDataSvcAptTradeDev/getRTMSDataSvcAptTradeDev",
    "https://apis.data.go.kr/1613000/RTMSDataSvcAptTrade/getRTMSDataSvcAptTrade",
]

def fetch_month(key, lawd, ym):
    last = ""
    for base in API_URLS:
        q = f"{base}?serviceKey={urllib.parse.quote(key, safe='') if '%' not in key else key}&LAWD_CD={lawd}&DEAL_YMD={ym}&pageNo=1&numOfRows=1000"
        try:
            raw = urllib.request.urlopen(q, timeout=20).read().decode("utf-8", "ignore")
        except Exception as e:
            last = str(e); continue
        root = ET.fromstring(raw)
        code = (root.findtext(".//resultCode") or "").strip()
        if code not in ("00", "000", ""):
            last = f"{code} {root.findtext('.//resultMsg')}"; continue
        rows = []
        for it in root.iter("item"):
            g = lambda *names: next(((it.findtext(n) or "").strip() for n in names if it.findtext(n) is not None), "")
            if g("cdealType") == "O":           # 해제된 거래는 뺍니다
                continue
            try:
                rows.append({"단지명": g("aptNm"), "동": g("umdNm"), "면적": float(g("excluUseAr")),
                             "층": g("floor"), "거래금액": int(g("dealAmount").replace(",", "")),
                             "거래일": dt.date(int(g("dealYear")), int(g("dealMonth")), int(g("dealDay"))),
                             "건축년도": g("buildYear")})
            except Exception:
                continue
        return rows, ""
    return [], last

def fetch_trades(key, region, months=12):
    lawd = REGIONS[region][0]
    rows, errs = [], []
    y, m = TODAY.year, TODAY.month
    for _ in range(months):
        r, e = fetch_month(key, lawd, f"{y}{m:02d}")
        rows += r
        if e: errs.append(e)
        m -= 1
        if m == 0: y, m = y - 1, 12
    df = pd.DataFrame(rows)
    if not df.empty:
        df["지역"] = region
    return df, errs

def med(series):
    s = [v for v in series if pd.notna(v)]
    return median(s) if s else None

def period_medians(df):
    out = {}
    for mo in (3, 6, 12):
        cut = TODAY - dt.timedelta(days=mo * 30)
        out[mo] = med(df[df["거래일"] >= cut]["거래금액"]) if not df.empty else None
    return out


# ---------- 4. 매물(호가): 사용자가 확인한 매물만 ----------
LISTING_HELP = "단지명, 전용면적(㎡), 층, 호가(만원), 매물 링크, 확인 날짜 — 한 줄에 매물 하나, 쉼표로 구분"

def parse_listings(text):
    rows, bad = [], []
    for i, line in enumerate((text or "").strip().splitlines(), 1):
        if not line.strip():
            continue
        p = [x.strip() for x in re.split(r"[,\t]", line)]
        if len(p) < 6:
            bad.append(i); continue
        try:
            rows.append({"단지명": p[0], "면적": num(p[1]), "층": p[2], "호가": int(num(p[3])),
                         "매물 링크": p[4], "확인 날짜": p[5]})
        except Exception:
            bad.append(i)
    return pd.DataFrame(rows), bad


# ---------- 5. 매수 절차 (공식 근거가 있는 일반 절차) ----------
def procedure_table(kind, permit, regulated, price):
    steps = []
    add = lambda *r: steps.append(list(r))
    if permit:
        add("토지거래허가 신청", "토지거래계약 허가신청서 · 토지이용계획서 · 취득자금 조달계획서 · 신분증", "부동산 소재지 시·군·구청", "신청 뒤 15일 이내 처분(법정)")
        add("허가증 받은 뒤 계약", "허가증 · 신분증 · 계약금", "매도인 · 공인중개사", "1일")
    if kind == "일반 아파트":
        add("등기부 · 건축물대장 · 토지이용계획 확인", "등기사항증명서 · 건축물대장", "인터넷등기소 · 정부24 · 토지이음", "1일")
        if not permit:
            add("매매계약 · 계약금 지급", "신분증 · 도장 · 계약금", "매도인 · 공인중개사", "1일")
    elif kind == "분양권":
        add("전매 제한 여부 · 분양계약서 확인", "분양계약서 · 전매 가능 확인", "시행사 · 관할 시·군·구청", "1~3일")
        add("분양권 매매계약 · 계약금 지급", "분양계약서 · 신분증 · 계약금", "매도인 · 공인중개사", "1일")
        add("시행사 명의변경", "매매계약서 · 거래신고필증 · 인감증명서", "시행사(분양사무소)", "1~7일")
    else:  # 재개발 · 재건축 입주권
        add("조합원 지위 양도 가능 여부 확인", "관리처분계획 · 조합원 자격 확인서", "조합 · 관할 시·군·구청", "1~7일")
        add("입주권 매매계약 · 계약금 지급", "신분증 · 계약금", "매도인 · 공인중개사", "1일")
        add("조합원 명의변경", "매매계약서 · 거래신고필증 · 인감증명서", "조합", "1~14일")
    add("부동산 거래신고", "매매계약서 · 신분증" + (" · 자금조달계획서" if (regulated or price >= 60000 or permit) else ""), "시·군·구청 또는 부동산거래관리시스템(rtms.molit.go.kr)", "계약 뒤 30일 이내")
    add("대출 신청 · 심사", "소득 증빙 · 재직증명 · 매매계약서", "은행", "2~4주")
    if kind == "일반 아파트":
        add("잔금 · 소유권이전등기", "잔금 · 등기 서류 · 취득세 영수증", "등기소(법무사)", "잔금일(이전등기는 잔금 뒤 60일 이내)")
    add("취득세 신고 · 납부", "취득세 신고서 · 매매계약서", "시·군·구청 세무과 · 위택스", "취득일부터 60일 이내")
    if permit:
        add("실거주 · 이용 의무 지키기", "전입신고", "주민센터", "허가 목적대로 이용(기간은 허가 조건 확인)")
    return pd.DataFrame([[i + 1] + s for i, s in enumerate(steps)],
                        columns=["순서", "진행 업무", "필요서류", "처리기관", "예상 소요기간"])

PERMIT_NOTE = f"""**토지거래허가가 필요할 때**
- 신청: 부동산이 있는 시·군·구청(허가 담당 부서)에 매도인 · 매수인이 함께 신청합니다.
- 서류: 토지거래계약 허가신청서, 토지이용계획서, 취득자금 조달계획서, 신분증.
- 처리기간: 신청을 받은 날부터 15일 이내에 허가 여부를 알려 줍니다(부동산 거래신고 등에 관한 법률).
- 허가 전 계약 주의: 허가 없이 맺은 계약은 효력이 없습니다(허가 전까지 「유동적 무효」). 허가 전에 큰 계약금을 보내지 마세요. 허가받으면 정해진 기간 동안 직접 살아야 해서, 세입자가 있는 집(갭투자)은 살 수 없습니다.
- 확인: 토지이음 https://www.eum.go.kr → 토지이용계획 열람에서 주소를 넣으면 「토지거래계약에 관한 허가구역」 표시를 볼 수 있습니다.

{UNSURE} 실거주 의무 기간 · 예외 조건 · 지정 기간은 지정 공고마다 다릅니다. 관할 시·군·구청 공고로 확인하세요."""


# ---------- 6. 단계별 비용표 (파이썬 계산) ----------
def cost_table(price, dep_pct, mid_pct, mid_n, d_contract, d_mid, d_balance, d_move, loan,
               houses_after, regulated, area, vat, reg_fee, move_fee):
    dep = price * dep_pct / 100
    mid_total = price * mid_pct / 100 if mid_n > 0 else 0
    bal = price - dep - mid_total
    if bal < 0:
        raise gr.Error("계약금 + 중도금 비율이 100% 를 넘습니다. 비율을 줄여 주세요.")
    if loan > bal:
        raise gr.Error(f"대출액({man(loan)})이 잔금({man(bal)})보다 큽니다. 대출은 잔금 때 실행되니 대출액을 줄이거나 잔금 비율을 늘려 주세요.")
    tax, rate = acquisition_tax(price, houses_after, regulated, area)
    fee = brokerage(price, vat)
    items = [(d_contract, "계약금", "매도인", dep, 0)]
    for i in range(int(mid_n)):
        items.append((d_mid + dt.timedelta(days=30 * i), f"중도금 {i+1}차", "매도인", mid_total / mid_n, 0))
    items += [(d_balance, "잔금", "매도인", bal, loan),
              (d_balance, f"취득세 등(세율 {rate}%)", "시·군·구청", tax, 0),
              (d_balance, "중개보수", "공인중개사", fee, 0),
              (d_balance, "등기비용(법무사 · 채권 등)", "법무사 · 등기소", reg_fee, 0),
              (d_move, "이사비", "이사업체", move_fee, 0)]
    items.sort(key=lambda x: x[0])
    rows, paid, loan_cum = [], 0.0, 0.0
    for i, (d, name, to, amt, ln) in enumerate(items, 1):
        paid += amt; loan_cum += ln
        rows.append([i, d.isoformat(), name, to, round(amt), round(paid), round(ln), round(paid - loan_cum)])
    df = pd.DataFrame(rows, columns=["순서", "지급 시점", "지급 항목", "지급 대상", "이번 지급액(만원)",
                                     "누적 지급액(만원)", "대출 실행액(만원)", "누적 자기자금(만원)"])
    return df, tax, fee


# ---------- 7. AI 의견: 지역 점수 · 장단점 ----------
REGION_PROMPT = """너는 대한민국 아파트 매수 상담가야. 아래 지역마다 자연환경 · 학군 · 교통 · 생활편의 · 미래가치를 1~5점으로 매기고
장점 · 단점을 한 문장씩 써줘. 모르면 점수를 3으로 두고 「잘 모름」이라고 써. 가격이나 숫자는 쓰지 마.
반드시 JSON 하나로만 답해: {"regions": [{"name": "지역 이름 그대로", "자연환경": 1~5, "학군": 1~5, "교통": 1~5, "생활편의": 1~5, "미래가치": 1~5, "장점": "...", "단점": "..."}]}"""

def ai_region_opinion(client, names):
    r = client.chat.completions.create(model=MODEL, response_format={"type": "json_object"},
        messages=[{"role": "system", "content": REGION_PROMPT},
                  {"role": "user", "content": "지역: " + ", ".join(names)}])
    data = json.loads(r.choices[0].message.content)
    return {x.get("name"): x for x in data.get("regions", [])}

def ai_web_check(client, question):
    """OpenAI 웹 검색 도구로 확인합니다. 안 되면 「데이터 접근 제한」을 돌려줍니다."""
    try:
        r = client.responses.create(model=MODEL, tools=[{"type": "web_search_preview"}], input=question)
        text = r.output_text
        links = []
        for o in r.output:
            for c in getattr(o, "content", None) or []:
                for a in getattr(c, "annotations", None) or []:
                    if getattr(a, "url", None):
                        links.append(f"- [{a.title or a.url}]({a.url})")
        if not links:
            return f"{UNSURE} 출처 링크가 없는 답이라 그대로 믿을 수 없습니다.\n\n{text}"
        return text + "\n\n**출처** (확인 날짜 " + TODAY.isoformat() + ")\n" + "\n".join(dict.fromkeys(links))
    except Exception as e:
        return f"**데이터 접근 제한** — 웹 검색을 쓸 수 없어 확인하지 못했습니다({type(e).__name__}). 지어내지 않고 비워 둡니다. 토지이음 https://www.eum.go.kr 과 관할 시·군·구청 공고로 직접 확인해 주세요."


# ---------- 상태 ----------
S = {"regions": pd.DataFrame(), "trades": pd.DataFrame(), "apts": pd.DataFrame(),
     "listings": pd.DataFrame(), "procedure": pd.DataFrame(), "cost": pd.DataFrame(), "summary": pd.DataFrame()}


# ---------- 페이지 1 → 2: 지역 추천 ----------
def run_regions(okey, mkey, cash, budget, income, exist_yearly, houses_now, wish, area, work,
                w1, w2, w3, w4, w5, w6, regulated, first_buyer, ltv_reg, ltv_free, ltv_first,
                dsr, rate, stress, years, cap_on):
    if not wish:
        raise gr.Error("1페이지에서 「희망 지역」을 1곳 이상 골라 주세요.")
    if not work:
        raise gr.Error("1페이지에서 「직장 위치」를 골라 주세요.")
    client = get_key(okey)
    houses_after = int(houses_now) + 1
    # 살 수 있는 최고 가격: 현금으로 비용(세금 · 수수료 약 4% 어림)과 대출 뺀 나머지를 냄
    best = 0
    for p in range(10000, 400001, 500):
        loan, _ = max_loan(p, income, exist_yearly, regulated, first_buyer, ltv_reg, ltv_free, ltv_first,
                           dsr, rate, stress, years, cap_on)
        tax, _ = acquisition_tax(p, houses_after, regulated, area)
        need = p - loan + tax + brokerage(p, True) + 300
        if need <= cash and p <= budget:
            best = p
    weights = dict(zip(FACTORS, [w1, w2, w3, w4, w5, w6]))
    op = ai_region_opinion(client, wish)
    rows, trades, msgs = [], [], []
    for name in wish:
        o = op.get(name, {})
        cm = commute_min(name, work)
        sc = {f: float(o.get(f, 3)) for f in FACTORS if f != "직주근접"}
        sc["직주근접"] = max(1.0, min(5.0, 5 - (cm - 30) / 15))      # 30분 이하 5점, 15분 늘 때마다 1점 깎음
        total = sum(sc[f] * weights[f] for f in FACTORS) / (5 * sum(weights.values())) * 100
        price_txt, med12 = "실거래 확인 불가", None
        if (mkey or "").strip():
            df, errs = fetch_trades(mkey.strip(), name)
            if not df.empty:
                band = df[(df["면적"] >= area - 5) & (df["면적"] <= area + 5)]
                med12 = med(band["거래금액"])
                price_txt = f"12개월 중앙값 {man(med12)} ({len(band)}건)" if med12 else "같은 면적 거래 없음"
                trades.append(df)
            elif errs:
                msgs.append(f"- {name}: 실거래가 조회 실패 ({errs[0]})")
        rows.append([name, round(total, 1), cm, f"{OPINION} {o.get('장점', '잘 모름')}",
                     f"{OPINION} {o.get('단점', '잘 모름')}", price_txt, med12,
                     REGIONS[name][1], REGIONS[name][2], *[sc[f] for f in FACTORS]])
    df = pd.DataFrame(rows, columns=["지역", "종합점수", "평균 출퇴근 시간(분)", "장점", "단점", "예상 시세",
                                     "_중앙값", "_위도", "_경도", *FACTORS])
    df = df.sort_values("종합점수", ascending=False).head(5).reset_index(drop=True)
    S["regions"] = df
    S["trades"] = pd.concat(trades, ignore_index=True) if trades else pd.DataFrame()
    show = df[["지역", "종합점수", "평균 출퇴근 시간(분)", "장점", "단점", "예상 시세"]]
    bar = px.bar(df, x="지역", y="종합점수", color="지역", text="종합점수", title="지역별 종합점수(100점)")
    bar.update_layout(showlegend=False)
    mp = (px.scatter_map if hasattr(px, "scatter_map") else px.scatter_mapbox)(
        df, lat="_위도", lon="_경도", text="지역", size=[12] * len(df), hover_name="지역",
        hover_data={"종합점수": True, "_위도": False, "_경도": False}, zoom=9, height=420,
        center={"lat": float(df["_위도"].mean()), "lon": float(df["_경도"].mean())})
    if not hasattr(px, "scatter_map"):
        mp.update_layout(mapbox_style="open-street-map")
    mp.update_layout(margin=dict(l=0, r=0, t=0, b=0))
    note = (f"**살 수 있는 최고 가격(어림): {man(best)}** — 보유 현금 {man(cash)}, 총예산 {man(budget)}, 대출 규정(1페이지)으로 파이썬이 계산했습니다.\n\n"
            f"- 출퇴근 시간은 직선거리로 어림한 값입니다. 실제 경로는 지도 앱으로 확인하세요.\n"
            f"- 장점 · 단점 · 환경 점수는 {OPINION} 입니다.\n"
            + ("- 국토교통부 API 키가 없어 실거래가는 비워 두었습니다." if not (mkey or '').strip() else "") + "\n" + "\n".join(msgs))
    return show, bar, mp, note, gr.update(choices=list(df["지역"]), value=df["지역"].iloc[0])


# ---------- 페이지 3: 아파트 · 매물 비교 ----------
def run_apartments(region, area, listings_text, max_price_man):
    tr = S["trades"]
    lst, bad = parse_listings(listings_text)
    S["listings"] = lst
    msg = []
    if bad:
        msg.append(f"매물 {', '.join(map(str, bad))}번째 줄은 형식이 맞지 않아 뺐습니다. ({LISTING_HELP})")
    rows = []
    if not tr.empty and region:
        d = tr[(tr["지역"] == region) & (tr["면적"] >= area - 5) & (tr["면적"] <= area + 5)]
        for apt, g in d.groupby("단지명"):
            pm = period_medians(g)
            m12 = pm[12]
            if m12 is None or (max_price_man and m12 > max_price_man):
                continue
            L = lst[lst["단지명"].str.replace(" ", "") == apt.replace(" ", "")] if not lst.empty else pd.DataFrame()
            ask_med = med(L["호가"]) if not L.empty else None
            if not L.empty:
                for _, x in L.iterrows():
                    rows.append([apt, x["면적"], x["층"], pm[3], pm[6], m12, ask_med, x["호가"],
                                 f"{x['매물 링크']} (확인 {x['확인 날짜']})", len(g)])
            else:
                rows.append([apt, round(g["면적"].median(), 1), "-", pm[3], pm[6], m12, None, "매물 확인 불가",
                             "국토교통부 실거래가 공개시스템 rt.molit.go.kr", len(g)])
        if d.empty:
            msg.append("이 지역 · 면적의 최근 12개월 실거래가 없습니다.")
    else:
        msg.append("실거래가가 없습니다. 1페이지에 국토교통부 API 키를 넣고 2페이지 「추천 받기」를 다시 눌러 주세요.")
    # 실거래와 짝이 안 맞는 매물도 그대로 보여 줍니다(실거래 중앙값은 비움)
    if not lst.empty:
        shown = {r[0].replace(" ", "") for r in rows}
        for _, x in lst.iterrows():
            if x["단지명"].replace(" ", "") not in shown:
                rows.append([x["단지명"], x["면적"], x["층"], None, None, None, x["호가"], x["호가"],
                             f"{x['매물 링크']} (확인 {x['확인 날짜']})", 0])
    df = pd.DataFrame(rows, columns=["단지명", "면적", "층수", "실거래 3개월 중앙값", "실거래 6개월 중앙값",
                                     "실거래 중앙값(12개월)", "매물 호가 중앙값", "매매가격", "출처", "거래 건수"])
    df = df.sort_values("거래 건수", ascending=False).head(5).reset_index(drop=True)
    S["apts"] = df
    view = df.copy()
    for c in ["실거래 3개월 중앙값", "실거래 6개월 중앙값", "실거래 중앙값(12개월)", "매물 호가 중앙값", "매매가격"]:
        view[c] = view[c].map(lambda v: man(v) if isinstance(v, (int, float)) and pd.notna(v) else (v if isinstance(v, str) else "-"))
    fig = go.Figure()
    if not df.empty:
        for c, nm in [("실거래 중앙값(12개월)", "실거래 중앙값"), ("매물 호가 중앙값", "호가 중앙값")]:
            fig.add_bar(x=df["단지명"], y=pd.to_numeric(df[c], errors="coerce"), name=nm)
        fig.update_layout(barmode="group", title="단지별 실거래 중앙값과 호가 중앙값 (만원)")
    if lst.empty:
        msg.append("매물(호가)을 붙여 넣지 않아 「매물 확인 불가」로 표시했습니다. 네이버부동산 등에서 직접 확인한 매물만 넣어 주세요.")
    choices = [f"{r['단지명']} · {r['면적']}㎡ · {r['매매가격'] if isinstance(r['매매가격'], str) else man(r['매매가격'])}" for _, r in df.iterrows()]
    return view.drop(columns=["거래 건수"]), fig, "\n\n".join(msg) or "비교를 마쳤습니다.", gr.update(choices=choices, value=choices[0] if choices else None)


# ---------- 페이지 4: 매수 절차 ----------
def run_procedure(okey, pick, kind, permit, adj, spec, do_web):
    price = 0
    if pick:
        row = S["apts"][S["apts"]["단지명"] == pick.split(" · ")[0]]
        if not row.empty and isinstance(row.iloc[0]["매매가격"], (int, float)):
            price = row.iloc[0]["매매가격"]
    df = procedure_table(kind, permit, adj or spec, price)
    S["procedure"] = df
    limits = []
    if permit: limits.append("토지거래허가구역: 허가 뒤 계약 · 실거주 의무 · 갭투자 불가")
    if adj: limits.append("조정대상지역: 다주택 취득세 중과 · LTV 낮아짐 · 자금조달계획서 제출")
    if spec: limits.append("투기과열지구: LTV 낮아짐 · 자금조달계획서와 증빙 제출 · 재건축 조합원 지위 양도 제한")
    head = f"**선택한 매물:** {pick or '없음'} · **매수 형태:** {kind}\n\n" + (
        "**적용되는 매수 제한**\n" + "\n".join("- " + x for x in limits) if limits else "규제 지역 표시를 고르지 않았습니다.") + \
        f"\n\n{UNSURE} 위 규제 지역 여부는 사용자가 고른 값입니다. 토지이음(eum.go.kr)과 국토교통부 · 관할 지자체 공고로 확인하세요."
    web = ""
    if do_web and pick:
        client = get_key(okey)
        region = S["regions"]["지역"].iloc[0] if not S["regions"].empty else ""
        web = "### 웹 검색 확인 결과\n" + ai_web_check(client,
              f"오늘({TODAY.isoformat()}) 기준 {region} {pick.split(' · ')[0]} 아파트가 토지거래허가구역, 조정대상지역, 투기과열지구에 해당하는지 "
              "국토교통부 · 지자체 · 금융위원회 공식 발표를 찾아 한국어로 짧게 알려줘. 확인 안 되면 확인 안 된다고 말해줘.")
    return head, df, (PERMIT_NOTE if permit else ""), web


# ---------- 페이지 5: 비용 대시보드 ----------
def run_cost(pick, price_in, area, dep_pct, mid_pct, mid_n, d_c, d_m, d_b, d_mv, loan_in, cash,
             income, exist_yearly, houses_now, regulated, first_buyer, ltv_reg, ltv_free, ltv_first,
             dsr, rate, stress, years, cap_on, vat, reg_fee, move_fee):
    price = num(price_in)
    if price <= 0:
        raise gr.Error("「매매가격(만원)」을 넣어 주세요. 3페이지에서 고른 매물의 가격을 그대로 써도 됩니다.")
    try:
        dates = [dt.date.fromisoformat(str(x).strip()) for x in (d_c, d_m, d_b, d_mv)]
    except ValueError:
        raise gr.Error("날짜는 2026-12-01 처럼 「연-월-일」로 적어 주세요.")
    loan_max, loan_df = max_loan(price, income, exist_yearly, regulated, first_buyer, ltv_reg, ltv_free, ltv_first,
                                 dsr, rate, stress, years, cap_on)
    loan = loan_max if num(loan_in) <= 0 else min(num(loan_in), loan_max)
    df, tax, fee = cost_table(price, dep_pct, mid_pct, mid_n, *dates, loan, int(houses_now) + 1,
                              regulated, area, vat, reg_fee, move_fee)
    S["cost"] = df
    total = df["이번 지급액(만원)"].sum()
    own = total - loan
    # 하루에 내 돈이 가장 많이 나가는 날 = 최대 현금 필요 시점
    day = df.assign(자기=df["이번 지급액(만원)"] - df["대출 실행액(만원)"]).groupby("지급 시점")["자기"].sum()
    peak_day, peak_amt = day.idxmax(), day.max()
    risks = []
    if own > cash: risks.append(f"보유 현금보다 {man(own - cash)} 더 필요합니다")
    if num(loan_in) > loan_max: risks.append(f"넣은 대출액이 예상 한도({man(loan_max)})보다 커서 한도로 낮췄습니다")
    if df["지급 시점"].min() < TODAY.isoformat(): risks.append("지난 날짜가 들어 있습니다")
    risks.append("대출 한도 · 세율은 기본값 기준입니다. 은행 · 위택스에서 다시 확인하세요")
    summ = pd.DataFrame([[pick or "직접 입력", man(total), man(own), man(loan),
                          f"{peak_day} · 이날 내 돈 {man(peak_amt)}",
                          " / ".join(risks)]],
                        columns=["추천 매물", "매수 총비용", "필요 현금", "예상 대출", "최대 자금 필요 시점", "매수 위험요인"])
    S["summary"] = summ
    fig = go.Figure()
    fig.add_bar(x=df["지급 시점"] + " " + df["지급 항목"], y=df["이번 지급액(만원)"], name="이번 지급액")
    fig.add_scatter(x=df["지급 시점"] + " " + df["지급 항목"], y=df["누적 지급액(만원)"], name="누적 지급액", mode="lines+markers")
    fig.add_scatter(x=df["지급 시점"] + " " + df["지급 항목"], y=df["누적 자기자금(만원)"], name="누적 자기자금", mode="lines+markers")
    fig.add_hline(y=cash, line_dash="dot", annotation_text="보유 현금")
    fig.update_layout(title="단계별 지출과 누적 금액 (만원)", height=420)
    return summ, df, fig, loan_df


def export_excel():
    if all(S[k].empty for k in ("regions", "apts", "procedure", "cost")):
        raise gr.Error("내보낼 결과가 없습니다. 2~5페이지를 먼저 실행해 주세요.")
    path = f"/tmp/부동산_매수분석_{TODAY.isoformat()}.xlsx" if os.path.isdir("/tmp") else f"부동산_매수분석_{TODAY.isoformat()}.xlsx"
    with pd.ExcelWriter(path, engine="openpyxl") as w:
        reg = S["regions"].drop(columns=[c for c in S["regions"].columns if c.startswith("_")], errors="ignore")
        for name, df in [("추천 지역", reg), ("매물 비교", S["apts"]), ("매수 절차", S["procedure"]),
                         ("비용 분석", S["cost"]), ("최종 결과", S["summary"])]:
            (df if not df.empty else pd.DataFrame({"안내": ["실행하지 않음"]})).to_excel(w, sheet_name=name, index=False)
    return path


# ---------- 화면 ----------
HOWTO = """## 시작하는 방법
1. **1페이지**에 돈 · 조건을 넣고, 맨 위에 **OpenAI API 키**를 넣습니다. 실거래가를 보려면 **국토교통부 API 키**(공공데이터포털)도 넣습니다.
2. **2페이지** 「추천 받기」 → 지역 점수 · 지도를 봅니다.
3. **3페이지**에 내가 직접 확인한 매물을 붙여 넣고 「비교하기」 → 실거래 중앙값과 호가를 비교합니다.
4. **4 · 5페이지**에서 절차와 비용을 봅니다. 숫자를 바꾸면 비용표가 바로 다시 계산됩니다. 맨 아래에서 엑셀로 받습니다.

계산은 모두 파이썬이 하고, AI 는 지역 장단점 같은 **의견**만 냅니다(【ChatGPT의 의견】 표시). 이 결과는 참고용이며 투자 권유가 아닙니다."""

with gr.Blocks(title="AI 부동산 매수 컨설턴트") as demo:
    gr.Markdown("# AI 부동산 매수 컨설턴트")
    gr.Markdown(HOWTO)
    with gr.Row():
        okey = gr.Textbox(label="OpenAI API 키", type="password", placeholder="sk-...")
        mkey = gr.Textbox(label="국토교통부 실거래가 API 키 (공공데이터포털, 없으면 비워 두기)", type="password")
    with gr.Tabs():
        with gr.Tab("1. 예산 · 조건"):
            with gr.Row():
                cash = gr.Number(label="보유 현금(만원)", value=40000)
                budget = gr.Number(label="총예산(만원)", value=90000)
                income = gr.Number(label="연봉(만원, 부부 합산 가능)", value=9000)
                exist_yearly = gr.Number(label="기존 대출 연간 원리금(만원)", value=0)
            with gr.Row():
                houses_now = gr.Dropdown([0, 1, 2, 3], value=0, label="지금 가진 주택 수")
                area = gr.Number(label="희망 전용면적(㎡)", value=84)
                work = gr.Dropdown(REGION_NAMES, label="직장 위치", value="서울 강남구")
            wish = gr.Dropdown(REGION_NAMES, multiselect=True, label="희망 지역(3~8곳 고르기)",
                               value=["경기 성남시 분당구", "경기 용인시 수지구", "경기 수원시 영통구", "경기 화성시"])
            gr.Markdown("**중요도 (1 = 덜 중요, 5 = 아주 중요)**")
            with gr.Row():
                ws = [gr.Slider(1, 5, value=3, step=1, label=f) for f in FACTORS]
            with gr.Accordion("대출 규정 (기본값을 확인하고 바꿀 수 있습니다)", open=False):
                gr.Markdown(f"{UNSURE} 기본값은 2025-10-15 주택시장 안정화 대책 기준입니다. 이후 바뀌었을 수 있으니 금융위원회 발표(fsc.go.kr)로 확인해 주세요.")
                with gr.Row():
                    regulated = gr.Checkbox(label="규제지역(조정 · 투기과열)에서 산다", value=True)
                    first_buyer = gr.Checkbox(label="생애최초 구입", value=False)
                    cap_on = gr.Checkbox(label="수도권 · 규제지역 금액 한도 적용(15억 이하 6억 · 25억 이하 4억 · 초과 2억)", value=True)
                with gr.Row():
                    ltv_reg = gr.Number(label="LTV 규제지역(%)", value=40)
                    ltv_free = gr.Number(label="LTV 비규제(%)", value=70)
                    ltv_first = gr.Number(label="LTV 생애최초(%)", value=70)
                    dsr = gr.Number(label="DSR(%)", value=40)
                with gr.Row():
                    rate = gr.Number(label="대출 금리(%)", value=4.0)
                    stress = gr.Number(label="스트레스 금리(%p)", value=3.0)
                    years = gr.Number(label="대출 기간(년)", value=30)
        with gr.Tab("2. 추천 지역 · 지도"):
            go_btn = gr.Button("추천 받기", variant="primary")
            reg_note = gr.Markdown()
            reg_tbl = gr.Dataframe(label="지역 추천", wrap=True)
            with gr.Row():
                reg_bar = gr.Plot(label="종합점수")
                reg_map = gr.Plot(label="지도")
        with gr.Tab("3. 아파트 · 매물 비교"):
            with gr.Row():
                region_pick = gr.Dropdown(label="지역", choices=[])
                max_price = gr.Number(label="최고 가격(만원, 0 이면 제한 없음)", value=0)
            listings = gr.Textbox(label="직접 확인한 매물 붙여 넣기", lines=5, placeholder=LISTING_HELP +
                                  "\n예) 래미안 OO, 84.9, 12/25, 98000, https://new.land.naver.com/..., 2026-11-09")
            apt_btn = gr.Button("비교하기", variant="primary")
            apt_note = gr.Markdown()
            apt_tbl = gr.Dataframe(label="매물 비교", wrap=True)
            apt_fig = gr.Plot()
        with gr.Tab("4. 매수 절차"):
            with gr.Row():
                pick = gr.Dropdown(label="매물 고르기(3페이지 결과)", choices=[])
                kind = gr.Radio(["일반 아파트", "분양권", "재개발 · 재건축 입주권"], value="일반 아파트", label="매수 형태")
            with gr.Row():
                permit = gr.Checkbox(label="토지거래허가구역")
                adj = gr.Checkbox(label="조정대상지역")
                spec = gr.Checkbox(label="투기과열지구")
                do_web = gr.Checkbox(label="웹 검색으로 규제 여부 확인(OpenAI)", value=False)
            proc_btn = gr.Button("절차 보기", variant="primary")
            proc_head = gr.Markdown()
            proc_tbl = gr.Dataframe(label="매수 절차", wrap=True)
            proc_permit = gr.Markdown()
            proc_web = gr.Markdown()
        with gr.Tab("5. 비용 · 자금 대시보드"):
            d0 = TODAY + dt.timedelta(days=7)
            with gr.Row():
                price_in = gr.Number(label="매매가격(만원)", value=90000)
                loan_in = gr.Number(label="대출액(만원, 0 이면 예상 한도만큼)", value=0)
                dep_pct = gr.Slider(5, 30, value=10, step=1, label="계약금 비율(%)")
                mid_pct = gr.Slider(0, 60, value=0, step=5, label="중도금 비율(%)")
                mid_n = gr.Dropdown([0, 1, 2, 3], value=0, label="중도금 횟수")
            with gr.Row():
                d_c = gr.Textbox(label="계약일", value=d0.isoformat())
                d_m = gr.Textbox(label="중도금 1차일", value=(d0 + dt.timedelta(days=30)).isoformat())
                d_b = gr.Textbox(label="잔금일", value=(d0 + dt.timedelta(days=75)).isoformat())
                d_mv = gr.Textbox(label="이사일", value=(d0 + dt.timedelta(days=76)).isoformat())
            with gr.Row():
                vat = gr.Checkbox(label="중개보수 부가세 10% 포함", value=True)
                reg_fee = gr.Number(label="등기비용 어림(만원)", value=150)
                move_fee = gr.Number(label="이사비(만원)", value=200)
            summ_tbl = gr.Dataframe(label="최종 결과", wrap=True)
            cost_fig = gr.Plot()
            cost_tbl = gr.Dataframe(label="비용 분석")
            loan_tbl = gr.Dataframe(label="대출 가능액 계산")
            gr.Markdown(f"{UNSURE} 취득세는 감면(생애최초 등)을 넣지 않은 기본 세율입니다. 정확한 금액은 위택스(wetax.go.kr)에서 확인하세요. 중개보수는 법정 상한 요율로 계산했습니다.")
    with gr.Row():
        xls_btn = gr.Button("엑셀로 받기")
        xls_file = gr.File(label="엑셀 파일")

    loan_inputs = [regulated, first_buyer, ltv_reg, ltv_free, ltv_first, dsr, rate, stress, years, cap_on]
    go_btn.click(run_regions, [okey, mkey, cash, budget, income, exist_yearly, houses_now, wish, area, work, *ws, *loan_inputs],
                 [reg_tbl, reg_bar, reg_map, reg_note, region_pick])
    apt_btn.click(run_apartments, [region_pick, area, listings, max_price], [apt_tbl, apt_fig, apt_note, pick])
    proc_btn.click(run_procedure, [okey, pick, kind, permit, adj, spec, do_web], [proc_head, proc_tbl, proc_permit, proc_web])

    def pick_price(p):
        row = S["apts"][S["apts"]["단지명"] == (p or "").split(" · ")[0]] if not S["apts"].empty else pd.DataFrame()
        v = row.iloc[0]["매매가격"] if not row.empty else None
        return gr.update(value=v) if isinstance(v, (int, float)) and pd.notna(v) else gr.update()
    pick.change(pick_price, pick, price_in)

    cost_inputs = [pick, price_in, area, dep_pct, mid_pct, mid_n, d_c, d_m, d_b, d_mv, loan_in, cash, income,
                   exist_yearly, houses_now, *loan_inputs, vat, reg_fee, move_fee]
    cost_outputs = [summ_tbl, cost_tbl, cost_fig, loan_tbl]
    for comp in [price_in, loan_in, dep_pct, mid_pct, mid_n, d_c, d_m, d_b, d_mv, vat, reg_fee, move_fee, pick]:
        comp.change(run_cost, cost_inputs, cost_outputs)       # 숫자를 바꾸면 바로 다시 계산
    demo.load(run_cost, cost_inputs, cost_outputs)
    xls_btn.click(export_excel, None, xls_file)

if __name__ == "__main__":
    demo.launch(share=True, debug=True)